# PRISM — LM Studio Conversation JSON to Q&A CSV

This notebook extracts question–answer (Q&A) pairs from an LM Studio chat conversation exported as JSON and saves them as a structured CSV file.

## Instructions

1. Export a conversation from LM Studio as a JSON file.
2. Place the JSON file in the same working directory as this notebook.
3. Run the code cell and select the JSON file by entering its displayed number.
4. The notebook writes `<input_filename>_QA.csv` with the columns `No`, `Question`, and `Answer`.
5. If some exchanges cannot be reliably paired, they are saved to `<input_filename>_unmatched.csv` for manual review.

**Requirements:** Python 3 and `pandas`. This notebook uses the LM Studio conversation JSON structure supported by the extraction script; other export formats may require adaptation. Review extracted pairs before using them for research evaluation.


In [ ]:

import json
import re
import csv
import pandas as pd
from pathlib import Path

# ==========================================
# Select a JSON file
# ==========================================

json_files = sorted(Path(".").glob("*.json"))

if not json_files:
    raise FileNotFoundError("No JSON files were found in the current directory.")

print("\n=== Available JSON files ===")
for i, path in enumerate(json_files, 1):
    print(f"[{i}] {path.name}")

while True:
    try:
        n = int(input("\nEnter the JSON file number: "))
        if 1 <= n <= len(json_files):
            break
    except ValueError:
        pass
    print("Please enter a valid number.")

json_path = json_files[n - 1]

with open(json_path, encoding="utf-8") as f:
    data = json.load(f)


# ==========================================
# Extract message text
# ==========================================

def content_text(items):
    texts = []
    for item in items or []:
        if not isinstance(item, dict):
            continue
        if item.get("type") == "text":
            value = item.get("text", "")
            if isinstance(value, str) and value.strip():
                texts.append(value.strip())
    return "\n".join(texts)


def get_user_text(version):
    return content_text(version.get("content", []))


def get_assistant_text(version):
    texts = []
    for step in version.get("steps", []):
        if step.get("type") == "contentBlock":
            t = content_text(step.get("content", []))
            if t:
                texts.append(t)
    return "\n".join(texts)


# ==========================================
# Parse Markdown tables
# ==========================================

def parse_table(text):
    rows = []

    for line in text.splitlines():
        line = line.strip()

        if not line.startswith("|"):
            continue

        cells = [
            c.strip().replace(r"\|", "|")
            for c in re.split(r"(?<!\\)\|", line.strip("|"))
        ]

        if not cells:
            continue

        if all(re.fullmatch(r":?-{3,}:?", c) for c in cells):
            continue

        if cells[0].lower() in ("no", "number", "#"):
            continue

        if cells[0].lower() in ("question", "質問"):
            continue

        if len(cells) >= 3 and cells[0].isdigit():
            rows.append({
                "No": int(cells[0]),
                "Question": cells[1],
                "Answer": " | ".join(cells[2:])
            })

        elif len(cells) == 2:
            rows.append({
                "No": None,
                "Question": cells[0],
                "Answer": cells[1]
            })

    return rows


# ==========================================
# Extract questions
# ==========================================

def extract_questions(text):
    questions = []

    # Prefer text following the ## Questions heading
    match = re.search(
        r"(?im)^#{1,6}\s*Questions\s*$",
        text
    )
    if match:
        text = text[match.end():]

    pattern = re.compile(
        r"^\s*(?:\*\*)?\s*"
        r"(?:Q\s*(\d*)\s*[:.]|(\d+)\s*[.):\t])"
        r"\s*(.+?)\s*(?:\*\*)?\s*$",
        re.IGNORECASE
    )

    for line in text.splitlines():
        m = pattern.match(line)
        if not m:
            continue

        number = m.group(1) or m.group(2)
        question = m.group(3).strip().strip("*")

        # Exclude numbered instructions by requiring a question mark
        if question.endswith("?"):
            questions.append({
                "No": int(number) if number else None,
                "Question": question
            })

    # Also support question-only tables
    if not questions:
        for row in parse_table(text):
            q = row["Question"]
            if q.endswith("?"):
                questions.append({
                    "No": row["No"],
                    "Question": q
                })

    return questions


# ==========================================
# Extract assistant answers
# ==========================================

def extract_answers(text):
    # 1. Markdown表
    table = parse_table(text)
    if table:
        return table

    # 2. **Q1: 質問** + 回答
    pattern = re.compile(
        r"(?m)^\s*\*\*Q\s*(\d*)\s*[:.]\s*(.+?)\*\*\s*\n"
        r"(.*?)"
        r"(?=^\s*\*\*Q\s*\d*\s*[:.]|\Z)",
        re.DOTALL | re.IGNORECASE
    )

    matches = pattern.findall(text)

    if matches:
        return [
            {
                "No": int(n) if n else None,
                "Question": q.strip(),
                "Answer": a.strip()
            }
            for n, q, a in matches
            if a.strip()
        ]

    # 3. Numbered answers
    pattern = re.compile(
        r"(?m)^\s*(?:\*\*)?\s*"
        r"(?:Q\s*)?(\d+)\s*[.):\t]\s*"
        r"(.*?)(?:\*\*)?\s*$"
    )

    lines = text.splitlines()
    answers = []
    current = None

    for line in lines:
        m = pattern.match(line)

        if m:
            if current:
                answers.append(current)

            current = {
                "No": int(m.group(1)),
                "Question": "",
                "Answer": m.group(2).strip()
            }

        elif current and line.strip():
            current["Answer"] += "\n" + line.strip()

    if current:
        answers.append(current)

    return answers


# ==========================================
# Process conversation messages
# ==========================================

rows = []
unmatched = []
pending_user_text = None

for message in data.get("messages", []):
    versions = message.get("versions", [])
    if not versions:
        continue

    selected = message.get("currentlySelected", 0)
    if not isinstance(selected, int) or not 0 <= selected < len(versions):
        selected = 0

    version = versions[selected]
    role = version.get("role")

    if role == "user":
        pending_user_text = get_user_text(version)
        continue

    if role != "assistant" or not pending_user_text:
        continue

    assistant_text = get_assistant_text(version)
    if not assistant_text:
        continue

    questions = extract_questions(pending_user_text)
    answers = extract_answers(assistant_text)

    # A. Answers include their own question text
    if answers and all(a["Question"] for a in answers):
        for a in answers:
            rows.append({
                "Question": a["Question"],
                "Answer": a["Answer"]
            })

    # B. Question and answer counts match
    elif questions and len(questions) == len(answers):
        for q, a in zip(questions, answers):
            rows.append({
                "Question": q["Question"],
                "Answer": a["Answer"]
            })

    # C. Single question with an unstructured answer
    elif len(questions) == 1 and not answers:
        rows.append({
            "Question": questions[0]["Question"],
            "Answer": assistant_text.strip()
        })

    # D. Match multiple questions and answers by number
    elif questions and answers:
        qmap = {
            q["No"]: q["Question"]
            for q in questions if q["No"] is not None
        }

        if (
            len(qmap) == len(questions)
            and all(a["No"] in qmap for a in answers)
            and len({a["No"] for a in answers}) == len(answers)
        ):
            for a in answers:
                rows.append({
                    "Question": qmap[a["No"]],
                    "Answer": a["Answer"]
                })
        else:
            unmatched.append({
                "User": pending_user_text,
                "Assistant": assistant_text
            })

    # E. Could not reliably pair questions and answers
    else:
        unmatched.append({
            "User": pending_user_text,
            "Assistant": assistant_text
        })

    pending_user_text = None


# ==========================================
# Export CSV files
# ==========================================

df = pd.DataFrame(
    rows,
    columns=["Question", "Answer"]
)

df.insert(0, "No", range(1, len(df) + 1))

output_path = json_path.with_name(
    json_path.stem + "_QA.csv"
)

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_MINIMAL
)

# Save unmatched conversations separately
if unmatched:
    unmatched_path = json_path.with_name(
        json_path.stem + "_unmatched.csv"
    )
    pd.DataFrame(unmatched).to_csv(
        unmatched_path,
        index=False,
        encoding="utf-8-sig"
    )


# ==========================================
# Display results
# ==========================================

print("\n=== Extraction results ===")
print(df.to_string(index=False))

print(f"\nQ&A pairs extracted: {len(df)}")
print(f"CSV: {output_path}")
print(f"Unmatched conversations: {len(unmatched)}")
